# Notebook 40 — Class-conditional frozen deep gate

This is a one-variable extension of the current `31_frozen_encoder_expert` winner. It reuses the exact Notebook 31 Stage-A encoder and Stage-B linear experts, freezes both throughout Stage C, retains the dropout-free `64 → 32 → 4` ReLU gate and `1e-3` Stage-C learning rate, and adds only a zero-initialized `4 experts × 22 classes` reliability matrix. The reference already exists and is never retrained here; only the 2,300 candidate parameters (2,212 gate + 88 reliability) are optimized.

The class-specific routing weights are `softmax_d(log g_d(x) + R[d,c])`, so zero reliability exactly reproduces ordinary dense soft routing at initialization.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
REFERENCE_PREFIX = 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_v1'
REFERENCE_CONFIG = 'config/compact_soft_moe_deeper_gate_frozen_stagec_3seed.yaml'
CANDIDATE_PREFIX = 'nfv3_4way_moe_soft_deeper_gate32_class_conditional_frozen_stagec_v1'
CANDIDATE_CONFIG = 'config/compact_soft_moe_class_conditional_deeper_gate_frozen_stagec_3seed.yaml'
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS = 2  # Matches the currently available paired evidence; use 3 only when reference seed 2 exists.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight and candidate execution

The preflight proves that the reference and candidate differ only by the class-conditional architecture/reliability term. Both reuse the same signed Stage-A/B artifacts. The candidate schedules Stage C alone, freezes the encoder and experts, and leaves exactly 2,300 parameters trainable.

In [ ]:
import copy, torch, yaml
from models.encoder import build_encoder
from training.logging_utils import run_streaming_logged
from training.model_utils import apply_stage_c_trainability, build_model
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
reference = yaml.safe_load(Path(REFERENCE_CONFIG).read_text())
candidate = yaml.safe_load(Path(CANDIDATE_CONFIG).read_text())
assert reference['seeds'] == candidate['seeds'] == [0, 1, 2]
assert reference['reuse_stage_a_from_prefix'] == candidate['reuse_stage_a_from_prefix'] == 'nfv3_4way_moe_soft_comparable_repro_v1'
assert reference['reuse_stage_b_from_prefix'] == candidate['reuse_stage_b_from_prefix'] == 'nfv3_4way_moe_soft_comparable_repro_v1'
ref = reference['backbone']; cand = candidate['backbone']
assert ref['architecture'] == 'moe_dataset_soft' and cand['architecture'] == 'moe_dataset_class_conditional'
for key in ('data', 'model', 'load_balance', 'evaluation'): assert ref[key] == cand[key]
ref_training = copy.deepcopy(ref['training']); cand_training = copy.deepcopy(cand['training'])
candidate_stage_c = cand_training['stage_c']; reference_stage_c = ref_training['stage_c']
assert candidate_stage_c['lambda_reliability'] == 0.0001
candidate_stage_c.pop('lambda_reliability')
assert ref_training == cand_training
assert reference_stage_c['optimizer']['lr'] == candidate_stage_c['optimizer']['lr'] == 0.001
assert cand['model']['gate']['hidden_dims'] == [32] and cand['model']['expert']['hidden_dims'] == []
assert cand['training']['stage_c_unfreeze'] == 'none' and cand['training']['stage_c']['freeze_experts'] is True
encoder = build_encoder(47, 64, cand['model']['encoder'])
model = build_model(cand['architecture'], encoder, ['UNSW', 'ToN', 'BoT', 'CIC'], [f'class_{i}' for i in range(22)], cand['model'])
apply_stage_c_trainability(model, cand)
assert model.class_reliability.shape == (4, 22)
assert torch.count_nonzero(model.class_reliability).item() == 0
assert sum(p.numel() for p in model.parameters()) == candidate['expected_total_parameters'] == 22420
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 2300
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', CANDIDATE_CONFIG, '--prefix', CANDIDATE_PREFIX]
run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label='40-dry-run')
if EXECUTE: run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label='40-training')
else: print('NO TRAINING WAS STARTED. Review the contract, set EXECUTE=True, and rerun this cell.')

## Paired validation-first comparison

This analysis intentionally works after any common completed seeds, rather than waiting for a three-seed `Final_Study_Manifest.json`. Validation macro-F1 remains the declared primary endpoint; accuracy, guardrails, and locked-test metrics show whether an apparent gain is broad or merely a different trade-off. Test metrics are descriptive and never select the winner.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
conditions = {'reference_frozen_deep_gate': REFERENCE_PREFIX, 'class_conditional_frozen_deep_gate': CANDIDATE_PREFIX}
summary_manifests = {name: (Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' / 'Final_Study_Manifest.json').is_file() for name, prefix in conditions.items()}
print('Three-seed summary completion:', summary_manifests)
validation_rows = []; test_rows = []; class_rows = []; dataset_rows = []; resource_rows = []
for condition, prefix in conditions.items():
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        required = ['manifest.json', 'Validation_Overall_Metrics.csv', 'Validation_Per_Dataset_Metrics.csv', 'Validation_Per_Class_Metrics.csv', 'Overall_Metrics.csv', 'Per_Class_Metrics.csv', 'Per_Dataset_Metrics.csv', 'Resource_Accounting.csv']
        if not all((result_dir / name).is_file() for name in required): continue
        overall_val = pd.read_csv(result_dir / 'Validation_Overall_Metrics.csv').iloc[0]
        val_ds = pd.read_csv(result_dir / 'Validation_Per_Dataset_Metrics.csv')
        val_cls = pd.read_csv(result_dir / 'Validation_Per_Class_Metrics.csv')
        rare = val_cls[(val_cls['support'] >= 20) & (val_cls['support'] < 200)]
        validation_rows.append({'condition': condition, 'seed': seed, 'validation_macro_f1': overall_val['macro_f1'], 'validation_accuracy': overall_val['accuracy'], 'worst_dataset_macro_f1': val_ds['macro_f1'].min(), 'rare_recall_mean': rare['recall'].mean() if len(rare) else np.nan, 'weakest_class_recall': val_cls.loc[val_cls['support'] > 0, 'recall'].min()})
        overall_test = pd.read_csv(result_dir / 'Overall_Metrics.csv').query("origin == 'ALL'").copy(); overall_test.insert(0, 'seed', seed); overall_test.insert(0, 'condition', condition); test_rows.append(overall_test)
        per_class = pd.read_csv(result_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL'").copy(); per_class.insert(0, 'seed', seed); per_class.insert(0, 'condition', condition); class_rows.append(per_class)
        per_dataset = pd.read_csv(result_dir / 'Per_Dataset_Metrics.csv').copy(); per_dataset.insert(0, 'seed', seed); per_dataset.insert(0, 'condition', condition); dataset_rows.append(per_dataset)
        resources = pd.read_csv(result_dir / 'Resource_Accounting.csv').copy(); resources['seed'] = seed; resources['condition'] = condition; resource_rows.append(resources)
validation = pd.DataFrame(validation_rows)
if validation.empty: print('No completed reference/candidate result pair exists yet.')
else:
    common = sorted(set(validation.query("condition == 'reference_frozen_deep_gate'")['seed']) & set(validation.query("condition == 'class_conditional_frozen_deep_gate'")['seed']))
    print('Completed paired seeds:', common)
    paired_validation = validation[validation['seed'].isin(common)].copy(); display(paired_validation)
    for metric in ['validation_macro_f1', 'validation_accuracy', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class_recall']:
        pivot = paired_validation.pivot(index='seed', columns='condition', values=metric)
        pivot['delta_conditional_minus_reference'] = pivot['class_conditional_frozen_deep_gate'] - pivot['reference_frozen_deep_gate']
        print(metric); display(pivot); print('Mean paired delta:', pivot['delta_conditional_minus_reference'].mean())
    if common and test_rows:
        test = pd.concat(test_rows, ignore_index=True); test = test[test['seed'].isin(common)]
        metrics = [c for c in ['accuracy', 'micro_f1', 'weighted_f1', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
        print('=== Locked-test descriptive metrics; never used for selection ==='); display(test.groupby('condition')[metrics].agg(['mean', 'std']))
        print('=== Per-dataset test macro-F1 ==='); display(pd.concat(dataset_rows, ignore_index=True).query('seed in @common').pivot_table(index='dataset', columns='condition', values='macro_f1', aggfunc='mean'))
        print('=== Per-class test F1 ==='); display(pd.concat(class_rows, ignore_index=True).query('seed in @common').pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    if resource_rows:
        resources = pd.concat(resource_rows, ignore_index=True); display(resources)
        candidate_resources = resources.query("condition == 'class_conditional_frozen_deep_gate'")
        assert set(candidate_resources['total_parameters'].astype(int)) == {22420}
        assert set(candidate_resources['trainable_parameters'].astype(int)) == {2300}

## Freeze integrity, routing, and learned reliability

The frozen encoder and experts must preserve Stage-B owned-expert validation metrics exactly. The diagnostics then show whether the candidate learned interpretable class-specific expert preferences or merely collapsed onto one expert.

In [ ]:
baseline_prefix = 'nfv3_4way_moe_soft_comparable_repro_v1'
for seed in (0, 1, 2):
    result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_seed{seed}'
    if not (result_dir / 'manifest.json').is_file(): continue
    stage_b_path = Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{baseline_prefix}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv'
    stage_c_path = result_dir / 'Validation_Expert_Owned_StageC.csv'
    stage_b = pd.read_csv(stage_b_path); stage_c = pd.read_csv(stage_c_path)
    merged = stage_b.merge(stage_c, on='dataset', suffixes=('_stage_b', '_stage_c'))
    delta_columns = []
    for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'):
        column = f'delta_{metric}'; merged[column] = merged[f'{metric}_stage_c'] - merged[f'{metric}_stage_b']; delta_columns.append(column)
    assert np.allclose(merged[delta_columns], 0.0, atol=1e-12), f'Frozen-body drift in seed {seed}'
    print(f'=== Seed {seed}: exact expert preservation ==='); display(merged)
    print(f'=== Seed {seed}: sample-level gate by source dataset ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
    print(f'=== Seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))
    reliability = pd.read_csv(result_dir / 'Class_Conditional_Reliability.csv')
    print(f'=== Seed {seed}: learned expert × class reliability ==='); display(reliability)
    history_path = result_dir / 'training' / 'Training_History.csv'
    if history_path.is_file():
        history = pd.read_csv(history_path).query("stage == 'C'")
        columns = [c for c in ['epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'train_reliability_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left'] if c in history]
        print(f'=== Seed {seed}: Stage-C history ==='); display(history[columns])

## Interpretation rule

Call the candidate an improvement only if its paired validation macro-F1 gain is repeatable and it does not introduce a material worst-dataset, rare-recall, or accuracy regression. A lower macro-F1 accompanied by higher weighted/sample-level test metrics is a trade-off, not a validation-selected win; locked-test results must not reverse the validation decision.